<a href="https://colab.research.google.com/github/naokityokoyama/fake_news_hdc/blob/main/Fake_news_Bert_Roberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unidecode evaluate num2words -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 6.6 MB/s eta 0:00:00


In [ ]:
import zipfile
import os
from unidecode import unidecode
import string
from num2words import num2words
import re
import numpy as np
import pandas as pd
from typing import Union, Literal
from tqdm.notebook import tqdm
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
import time
import re
#import gensim.downloader as api
import joblib
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#build dataset
def build_dataset(isot, covid, fever):
  if isot:
    df = pd.read_csv('/content/drive/MyDrive/uff/isot.csv')
    return df.sample(frac=1, random_state=42).reset_index(drop=True)
  elif covid:
    df = pd.read_csv('/content/drive/MyDrive/uff/covid.csv')
    return df.sample(frac=1, random_state=42).reset_index(drop=True)
  elif fever:
    df = pd.read_csv('/content/drive/MyDrive/uff/fever.csv')
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df = build_dataset(isot=True, covid=False, fever=False)
X_T, X_t, y_T, y_t = train_test_split(df['frase'], df['target'], test_size=0.30, random_state = 42)

In [ ]:
def batch(dataset, batch=True, size=5000):
  # Definir o tamanho da amostra
  if batch:
    sample_size = size

    # Criar uma amostra balanceada
    dataset = dataset.groupby("target", group_keys=False).apply(lambda x: resample(x, n_samples=sample_size // dataset["target"].nunique(), random_state=42))
    dataset = dataset.reset_index(drop=True)
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    return dataset

  else:
    return dataset

In [ ]:
def n2w(texto:str)->str:
  padrao = r"\d+"
  numeros = re.findall(padrao, texto)
  for numero in numeros:
    # Check if the number is within the num2words limit
    if abs(int(numero)) < 10**27: # The limit is 10^27 for num2words
        try:
            extenso = num2words(numero, lang='pt')
            texto = texto.replace(numero, extenso)
        except ValueError: # Handle potential errors during conversion
            pass # Keep the original number if conversion fails
    else:
        texto = texto.replace(numero, "[LARGE_NUMBER]") # Replace very large numbers with a placeholder
  return texto

In [ ]:
for repet in tqdm(range(2)):  #bug para rodar 2x
  df['frase'] = df['frase'].str.lower()
  df['frase'] = df['frase'].str.replace(f"[{string.punctuation}]", "", regex=True)
  df['frase'] = df['frase'].apply(lambda x: ' '.join(x.split()))
  df['frase'] = df['frase'].str.replace('"', '').str.replace('\\', '')
  #df['frase'] = df['frase'].apply(n2w)
  df['frase'] = df['frase'].apply(unidecode)

In [ ]:
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
df_ = df[['frase', 'target']]

In [ ]:
df_ = df_.rename(columns={'target': 'label'})

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoConfig
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np

# Garantindo que df_ esteja definido
df_ = df[['frase', 'target']].rename(columns={'target': 'label'})

train_df, val_df = train_test_split(df_, test_size=0.3, random_state=42)

hf_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df, preserve_index=False)
})

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return clf_metrics.compute(predictions=predictions, references=labels)

def train_and_evaluate_model(model_name, dataset, output_dir):
    print(f"\n{'='*50}")
    print(f"Iniciando pipeline para o modelo: {model_name}")
    print(f"{'='*50}\n")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, force_download=True)

    def tokenize_function(examples):
        return tokenizer(examples["frase"], truncation=True, padding=False, max_length=128)

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True,
        force_download=True
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        push_to_hub=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"Treinando {model_name}...")
    trainer.train()

    print(f"\nAvaliação final do {model_name}:")
    eval_results = trainer.evaluate()

    predictions_output = trainer.predict(tokenized_datasets["validation"])

    y_pred = np.argmax(predictions_output.predictions, axis=1)
    y_prob = torch.nn.functional.softmax(torch.tensor(predictions_output.predictions), dim=-1).numpy()[:, 1]
    y_true = predictions_output.label_ids

    cm = confusion_matrix(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)
    eval_results['confusion_matrix'] = cm
    eval_results['roc_auc'] = roc

    return eval_results


In [ ]:
modelos_para_testar = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base"
}

resultados_finais = {}

for nome_amigavel, identificador_hf in modelos_para_testar.items():
    diretorio_saida = f"./resultados_{nome_amigavel.lower()}"

    metricas = train_and_evaluate_model(
        model_name=identificador_hf,
        dataset=hf_dataset,
        output_dir=diretorio_saida
    )

    resultados_finais[nome_amigavel] = metricas

# Exibir Relatório Final
print("\n" + "="*50)
print("COMPARAÇÃO DE RESULTADOS")
print("="*50)
for modelo, metricas in resultados_finais.items():
    print(f"Modelo: {modelo}")
    print(f"  - Acurácia: {metricas.get('eval_accuracy', 0):.4f}")
    print(f"  - Precisão: {metricas.get('eval_precision', 0):.4f}")
    print(f"  - Recall:   {metricas.get('eval_recall', 0):.4f}")
    print(f"  - F1-Score: {metricas.get('eval_f1', 0):.4f}")
    if 'roc_auc' in metricas:
        print(f"  - ROC AUC:  {metricas.get('roc_auc', 0):.4f}")
    print("\n")

    # Imprimir a matriz de confusão
    print("  - Matriz de Confusão:")
    cm = metricas.get('confusion_matrix')
    if cm is not None:
        # Formatação simples e visual
        print(f"      [ {cm[0][0]:4d} | {cm[0][1]:4d} ] (Reais: 0)")
        print(f"      [ {cm[1][0]:4d} | {cm[1][1]:4d} ] (Reais: 1)")
        print("        (Pred:0) (Pred:1)")
    else:
        print("      Não disponível")
    print("\n")


In [ ]:
import pandas as pd

# Exibindo detalhadamente a configuração da Matriz de Confusão e ROC AUC para BERT e RoBERTa
for modelo, metricas in resultados_finais.items():
    print(f"--- Configuração de Matriz para {modelo} ---")
    if 'confusion_matrix' in metricas:
        cm = metricas['confusion_matrix']
        display(pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1']))
    else:
        print(f"A matriz de confusão não foi encontrada nos resultados de {modelo}.")

    if 'roc_auc' in metricas:
        print(f"ROC AUC: {metricas['roc_auc']:.4f}")
    print('\n')
